# Official Federated LSTM

Ovaj notebook je sluzbeni execution path za federirano ucenje LSTM modela nad EV charging forecasting splitovima.
Zadrzava isti `LSTMRegressor`, istu evaluacijsku logiku i isti artifact stil kao centralized LSTM notebook, ali trenira globalni model kroz FedAvg ili FedProx.

## Sto se ovdje dogada

Notebook prolazi kroz cijeli federirani workflow:

1. Ucitava federirane `.npz` splitove po klijentima.
2. Validira da svaki split ima oblik `[N, 24, 19]` i da nema NaN/Inf vrijednosti.
3. Trenera lokalne LSTM modele na privatnom `train.npz` splitu svakog klijenta.
4. Agregira lokalne tezine ponderiranim FedAvg pravilom.
5. Evaluira globalni model na spojenim train/val/test splitovima bez mijesanja splitova.
6. Sprema metrike, checkpoint, predikcije i osnovne grafove.

## 1. Configuration

Parametri za brzi debug run su na jednom mjestu. Za puni eksperiment povecaj `NUM_ROUNDS` i postavi `DEBUG_MAX_CLIENTS = None`.

In [19]:
ALGORITHM = "fedavg"  # "fedavg" or "fedprox"
SCENARIO = "iid"  # "iid" or "non_iid"

NUM_ROUNDS = 100
LOCAL_EPOCHS = 3
BATCH_SIZE = 256
LR = 0.001
WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0
FEDPROX_MU = 0.01
SEED = 42

HIDDEN_SIZE = 128
NUM_LAYERS = 1
DROPOUT = 0.0
DEVICE_REQUEST = "auto"

# None means auto-detect from SCENARIO. You can set an explicit path string here.
FEDERATED_DATASET_DIR = None
OUTPUT_ROOT = None
RUN_NAME = None

# Initial debug default: use only first 5 clients. Set to None for all clients.
DEBUG_MAX_CLIENTS = None

## 2. Imports and paths

Prvo nalazimo project root, dodajemo ga u `sys.path`, pa importamo postojece projektne module.

In [20]:
import json
import os
import random
import sys
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, TensorDataset

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "diplomski").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "diplomski").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    fallback_root = Path("/home/azureuser/cloudfiles/code/Users/lgjuric/federated-ev-charging-forecasting").resolve()
    if not (fallback_root / "diplomski").exists():
        raise RuntimeError("Run this notebook from the project root or from notebooks/.")
    PROJECT_ROOT = fallback_root

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from diplomski.evaluation import (
    SUPPORTED_FEATURE_COUNT,
    SUPPORTED_HORIZON,
    SUPPORTED_TARGET_COLUMN,
    SUPPORTED_WINDOW_SIZE,
    build_prediction_frame,
    compute_regression_metrics,
    configure_run_logger,
    load_experiment_metadata,
    make_run_name,
    validate_official_task,
)
from diplomski.models import LSTMRegressor
from diplomski.preprocessing.utils import save_json

try:
    from diplomski.evaluation import evaluate_split as project_evaluate_split
except ImportError:
    project_evaluate_split = None

ALGORITHM = ALGORITHM.strip().lower()
SCENARIO = SCENARIO.strip().lower()
if ALGORITHM not in {"fedavg", "fedprox"}:
    raise ValueError("ALGORITHM must be 'fedavg' or 'fedprox'.")
if SCENARIO not in {"iid", "non_iid"}:
    raise ValueError("SCENARIO must be 'iid' or 'non_iid'.")

METADATA_PATH = Path(
    os.environ.get(
        "DIPLOMSKI_EXPERIMENT_METADATA",
        str(PROJECT_ROOT / "data" / "processed" / "experiments" / "artifacts" / "experiment_metadata.json"),
    )
).resolve()

if FEDERATED_DATASET_DIR is None:
    env_dataset_dir = os.environ.get("DIPLOMSKI_FL_DATASET_DIR")
    FEDERATED_DATASET_DIR = Path(env_dataset_dir).resolve() if env_dataset_dir else None
else:
    FEDERATED_DATASET_DIR = Path(FEDERATED_DATASET_DIR).resolve()

if OUTPUT_ROOT is None:
    OUTPUT_ROOT = Path(
        os.environ.get(
            "DIPLOMSKI_FL_OUTPUT_ROOT",
            str(PROJECT_ROOT / "data" / "processed" / "results" / "federated"),
        )
    ).resolve()
else:
    OUTPUT_ROOT = Path(OUTPUT_ROOT).resolve()

if RUN_NAME is None:
    RUN_NAME = os.environ.get("DIPLOMSKI_FL_RUN_NAME", f"notebook_{ALGORITHM}_{SCENARIO}_{make_run_name()}")

run_dir = OUTPUT_ROOT / RUN_NAME
plots_dir = run_dir / "plots"
run_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"ALGORITHM={ALGORITHM} SCENARIO={SCENARIO}")
print(f"METADATA_PATH={METADATA_PATH}")
print(f"OUTPUT_ROOT={OUTPUT_ROOT}")
print(f"RUN_NAME={RUN_NAME}")

PROJECT_ROOT=/mnt/batch/tasks/shared/LS_root/mounts/clusters/lgjuric2/code/Users/lgjuric/federated-ev-charging-forecasting
ALGORITHM=fedavg SCENARIO=iid
METADATA_PATH=/mnt/batch/tasks/shared/LS_root/mounts/clusters/lgjuric2/code/Users/lgjuric/federated-ev-charging-forecasting/data/processed/experiments/artifacts/experiment_metadata.json
OUTPUT_ROOT=/mnt/batch/tasks/shared/LS_root/mounts/clusters/lgjuric2/code/Users/lgjuric/federated-ev-charging-forecasting/data/processed/results/federated
RUN_NAME=notebook_fedavg_iid_20260511T184320Z


## 3. Data structures

`SplitArrays` opisuje jedan `.npz` split, a `ClientDatasetBundle` drzi tri splita jednog federiranog klijenta.

In [21]:
@dataclass(frozen=True)
class SplitArrays:
    X: np.ndarray
    y: np.ndarray
    t: np.ndarray
    station_ids: np.ndarray


@dataclass(frozen=True)
class ClientDatasetBundle:
    client_id: str
    train: SplitArrays
    val: SplitArrays
    test: SplitArrays

## 4. Loading functions

Ove funkcije su namjerno stroge oko sheme i oblika podataka. Prazni splitovi su dozvoljeni, ali se preskacu kod globalne evaluacije.

In [22]:
def resolve_federated_dataset_dir(project_root: Path, scenario: str, configured_dir: Path | None) -> Path:
    if configured_dir is not None:
        if not configured_dir.exists():
            raise FileNotFoundError(f"Configured FEDERATED_DATASET_DIR does not exist: {configured_dir}")
        return configured_dir

    candidates_by_scenario = {
        "iid": [
            project_root / "data" / "processed" / "experiments" / "federated" / "iid_40_clients",
            project_root / "data" / "processed" / "experiments" / "iid",
        ],
        "non_iid": [
            project_root / "data" / "processed" / "experiments" / "federated" / "non_iid_station",
            project_root / "data" / "processed" / "experiments" / "non_iid_station",
        ],
    }
    for candidate in candidates_by_scenario[scenario]:
        if candidate.exists():
            return candidate.resolve()
    searched = "\n".join(str(path) for path in candidates_by_scenario[scenario])
    raise FileNotFoundError(f"Could not find federated dataset directory. Searched:\n{searched}")


def load_split_npz(
    path: Path,
    split_name: str,
    expected_window_size: int,
    expected_feature_count: int,
) -> SplitArrays:
    if not path.exists():
        raise FileNotFoundError(f"Missing {split_name} split file: {path}")

    required_keys = {"X", "y", "t", "station_ids"}
    with np.load(path) as data:
        missing_keys = required_keys.difference(data.files)
        if missing_keys:
            raise RuntimeError(
                f"{split_name} split at {path} is missing keys {sorted(missing_keys)}. "
                f"Available keys: {sorted(data.files)}."
            )
        X = data["X"].astype(np.float32, copy=False)
        y = data["y"].astype(np.float32, copy=False)
        t = data["t"].astype(str)
        station_ids = data["station_ids"].astype(str)

    if X.ndim != 3:
        raise RuntimeError(f"{split_name} X must be 3D, got shape {X.shape} in {path}.")
    if y.ndim != 1:
        raise RuntimeError(f"{split_name} y must be 1D, got shape {y.shape} in {path}.")
    if X.shape[1] != expected_window_size:
        raise RuntimeError(
            f"{split_name} X has window size {X.shape[1]}, expected {expected_window_size} in {path}."
        )
    if X.shape[2] != expected_feature_count:
        raise RuntimeError(
            f"{split_name} X has feature count {X.shape[2]}, expected {expected_feature_count} in {path}."
        )
    if not (len(X) == len(y) == len(t) == len(station_ids)):
        raise RuntimeError(
            f"{split_name} arrays must agree on sample count in {path}: "
            f"len(X)={len(X)}, len(y)={len(y)}, len(t)={len(t)}, len(station_ids)={len(station_ids)}."
        )
    if not np.isfinite(X).all() or not np.isfinite(y).all():
        raise RuntimeError(f"{split_name} split contains NaN or Inf values: {path}")

    return SplitArrays(X=X, y=y, t=t, station_ids=station_ids)


def load_federated_clients(
    federated_dir: Path,
    expected_window_size: int,
    expected_feature_count: int,
    skip_empty_val_test: bool = False,
) -> list[ClientDatasetBundle]:
    if not federated_dir.exists():
        raise FileNotFoundError(f"Federated dataset directory not found: {federated_dir}")

    client_dirs = sorted(path for path in federated_dir.iterdir() if path.is_dir() and path.name.startswith("client_"))
    if not client_dirs:
        raise FileNotFoundError(f"No client_* directories found in {federated_dir}")

    clients: list[ClientDatasetBundle] = []
    for client_dir in client_dirs:
        splits = {
            split_name: load_split_npz(
                client_dir / f"{split_name}.npz",
                f"{client_dir.name}/{split_name}",
                expected_window_size,
                expected_feature_count,
            )
            for split_name in ("train", "val", "test")
        }
        if skip_empty_val_test and (len(splits["val"].y) == 0 or len(splits["test"].y) == 0):
            continue
        clients.append(
            ClientDatasetBundle(
                client_id=client_dir.name,
                train=splits["train"],
                val=splits["val"],
                test=splits["test"],
            )
        )

    if not clients:
        raise RuntimeError(f"No usable clients found in {federated_dir}")
    return clients


def concat_splits(clients: list[ClientDatasetBundle], split_name: str) -> SplitArrays | None:
    if split_name not in {"train", "val", "test"}:
        raise ValueError("split_name must be one of: train, val, test.")

    non_empty_splits = [getattr(client, split_name) for client in clients if len(getattr(client, split_name).y) > 0]
    if not non_empty_splits:
        return None

    return SplitArrays(
        X=np.concatenate([split.X for split in non_empty_splits], axis=0).astype(np.float32, copy=False),
        y=np.concatenate([split.y for split in non_empty_splits], axis=0).astype(np.float32, copy=False),
        t=np.concatenate([split.t for split in non_empty_splits], axis=0).astype(str),
        station_ids=np.concatenate([split.station_ids for split in non_empty_splits], axis=0).astype(str),
    )

## Shared utility functions

Ovo su male pomocne funkcije za seed, uredjaj i PyTorch DataLoader. Iste su po namjeri kao u centralized notebooku.

In [23]:
def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def select_device(device_name: str) -> torch.device:
    normalized = device_name.strip().lower()
    if normalized == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if normalized == "cpu":
        return torch.device("cpu")
    if normalized == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA was requested but is not available.")
        return torch.device("cuda")
    raise ValueError(f"Unsupported device '{device_name}'. Use auto, cpu, or cuda.")


def build_dataloader(
    split_arrays: SplitArrays,
    batch_size: int,
    *,
    shuffle: bool,
    device: torch.device,
) -> DataLoader:
    if len(split_arrays.y) == 0:
        raise ValueError("Cannot build a DataLoader for an empty split.")
    dataset = TensorDataset(torch.from_numpy(split_arrays.X), torch.from_numpy(split_arrays.y))
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        pin_memory=device.type == "cuda",
    )


def count_trainable_parameters(model: nn.Module) -> int:
    return int(sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))


def estimate_model_state_bytes(model: nn.Module) -> int:
    state_dict = model.state_dict()
    return int(sum(tensor.numel() * tensor.element_size() for tensor in state_dict.values()))

## 5. Data validation summary

Nakon ucitavanja klijenata spremamo tablicu s brojem uzoraka, brojem stanica u train splitu i vremenskim granicama po splitu.

In [24]:
def timestamp_bounds(values: np.ndarray) -> tuple[str | None, str | None]:
    if len(values) == 0:
        return None, None
    parsed = pd.to_datetime(pd.Series(values), errors="coerce", utc=True)
    if parsed.notna().any():
        return parsed.min().isoformat(), parsed.max().isoformat()
    as_strings = pd.Series(values).astype(str)
    return str(as_strings.min()), str(as_strings.max())


def build_client_summary(clients: list[ClientDatasetBundle]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for client in clients:
        row: dict[str, Any] = {
            "client_id": client.client_id,
            "train_samples": int(len(client.train.y)),
            "val_samples": int(len(client.val.y)),
            "test_samples": int(len(client.test.y)),
            "train_station_count": int(pd.Series(client.train.station_ids).nunique()) if len(client.train.y) else 0,
        }
        for split_name in ("train", "val", "test"):
            split_arrays = getattr(client, split_name)
            min_t, max_t = timestamp_bounds(split_arrays.t)
            row[f"{split_name}_min_t"] = min_t
            row[f"{split_name}_max_t"] = max_t
        rows.append(row)
    return pd.DataFrame(rows)


metadata = load_experiment_metadata(METADATA_PATH)
validate_official_task(metadata, expected_feature_count=SUPPORTED_FEATURE_COUNT)
EXPECTED_WINDOW_SIZE = int(metadata.get("window_size", SUPPORTED_WINDOW_SIZE))
EXPECTED_FEATURE_COUNT = int(metadata.get("feature_count", SUPPORTED_FEATURE_COUNT))
FEDERATED_DATASET_DIR = resolve_federated_dataset_dir(PROJECT_ROOT, SCENARIO, FEDERATED_DATASET_DIR)

all_clients = load_federated_clients(
    FEDERATED_DATASET_DIR,
    expected_window_size=EXPECTED_WINDOW_SIZE,
    expected_feature_count=EXPECTED_FEATURE_COUNT,
    skip_empty_val_test=(SCENARIO == "non_iid"),
)
if DEBUG_MAX_CLIENTS is not None:
    all_clients = all_clients[: int(DEBUG_MAX_CLIENTS)]

train_clients = [client for client in all_clients if len(client.train.y) > 0]
if not train_clients:
    raise RuntimeError("No clients with non-empty train splits were found.")

global_train = concat_splits(train_clients, "train")
global_val = concat_splits(all_clients, "val")
global_test = concat_splits(all_clients, "test")
if global_train is None:
    raise RuntimeError("Global train split is empty after concatenation.")
if global_val is None:
    raise RuntimeError("At least one non-empty validation split is required for model selection.")

client_summary = build_client_summary(all_clients)
client_summary.to_csv(run_dir / "client_summary.csv", index=False)

selected_device = select_device(DEVICE_REQUEST)
set_global_seed(SEED)
logger = configure_run_logger(
    name=f"diplomski.notebooks.federated_lstm.{RUN_NAME}",
    log_path=run_dir / "run.log",
)

print(f"FEDERATED_DATASET_DIR={FEDERATED_DATASET_DIR}")
print(f"selected_device={selected_device}")
print(f"loaded_clients={len(all_clients)} train_clients={len(train_clients)}")
print(
    "global samples | "
    f"train={len(global_train.y)} val={len(global_val.y)} "
    f"test={0 if global_test is None else len(global_test.y)}"
)

client_summary

FEDERATED_DATASET_DIR=/mnt/batch/tasks/shared/LS_root/mounts/clusters/lgjuric2/code/Users/lgjuric/federated-ev-charging-forecasting/data/processed/experiments/iid
selected_device=cpu
loaded_clients=40 train_clients=40
global samples | train=110642 val=6824 test=30275


,client_id,train_samples,val_samples,test_samples,train_station_count,train_min_t,train_max_t,val_min_t,val_max_t,test_min_t,test_max_t
0,client_000,2767,171,757,40,2022-09-02T00:00:00+00:00,2023-01-05T15:00:00+00:00,2023-01-06T16:00:00+00:00,2023-01-13T18:00:00+00:00,2023-01-15T17:00:00+00:00,2023-02-21T08:00:00+00:00
1,client_001,2767,171,757,40,2022-09-02T00:00:00+00:00,2023-01-05T15:00:00+00:00,2023-01-06T16:00:00+00:00,2023-01-14T16:00:00+00:00,2023-01-15T17:00:00+00:00,2023-02-25T07:00:00+00:00
2,client_002,2766,171,757,40,2022-09-02T00:00:00+00:00,2023-01-05T15:00:00+00:00,2023-01-06T16:00:00+00:00,2023-01-14T16:00:00+00:00,2023-01-25T02:00:00+00:00,2023-02-25T14:00:00+00:00
3,client_003,2766,171,757,40,2022-09-02T03:00:00+00:00,2023-01-05T13:00:00+00:00,2023-01-07T02:00:00+00:00,2023-01-14T16:00:00+00:00,2023-01-15T17:00:00+00:00,2023-02-28T23:00:00+00:00
4,client_004,2766,171,757,40,2022-09-02T02:00:00+00:00,2023-01-05T14:00:00+00:00,2023-01-06T16:00:00+00:00,2023-01-14T13:00:00+00:00,2023-01-15T17:00:00+00:00,2023-02-28T23:00:00+00:00
5,client_005,2766,171,757,40,2022-09-02T00:00:00+00:00,2023-01-05T15:00:00+00:00,2023-01-06T16:00:00+00:00,2023-01-14T16:00:00+00:00,2023-01-23T13:00:00+00:00,2023-02-24T01:00:00+00:00
6,client_006,2766,171,757,40,2022-09-02T00:00:00+00:00,2023-01-05T13:00:00+00:00,2023-01-06T16:00:00+00:00,2023-01-14T16:00:00+00:00,2023-01-15T17:00:00+00:00,2023-02-28T23:00:00+00:00
7,client_007,2766,171,757,40,2022-09-02T00:00:00+00:00,2023-01-05T15:00:00+00:00,2023-01-06T16:00:00+00:00,2023-01-14T16:00:00+00:00,2023-01-15T17:00:00+00:00,2023-02-28T23:00:00+00:00
8,client_008,2766,171,757,40,2022-09-02T01:00:00+00:00,2023-01-05T13:00:00+00:00,2023-01-06T16:00:00+00:00,2023-01-14T16:00:00+00:00,2023-01-15T17:00:00+00:00,2023-02-28T23:00:00+00:00
9,client_009,2766,171,757,40,2022-09-02T00:00:00+00:00,2023-01-05T14:00:00+00:00,2023-01-06T16:00:00+00:00,2023-01-14T16:00:00+00:00,2023-01-16T20:00:00+00:00,2023-02-17T08:00:00+00:00


## Save configuration

Konfiguracija runa sprema se prije treninga kako bi artefakti bili reproducibilni.

In [25]:
def save_training_config(output_path: Path, selected_device: torch.device) -> None:
    payload = {
        "algorithm": ALGORITHM,
        "scenario": SCENARIO,
        "dataset_dir": str(FEDERATED_DATASET_DIR),
        "experiment_metadata_path": str(METADATA_PATH),
        "output_root": str(OUTPUT_ROOT),
        "run_dir": str(run_dir),
        "run_name": RUN_NAME,
        "device": str(selected_device),
        "seed": SEED,
        "debug_max_clients": DEBUG_MAX_CLIENTS,
        "model": {
            "input_size": EXPECTED_FEATURE_COUNT,
            "hidden_size": HIDDEN_SIZE,
            "num_layers": NUM_LAYERS,
            "dropout": DROPOUT,
        },
        "federated": {
            "num_rounds": NUM_ROUNDS,
            "local_epochs": LOCAL_EPOCHS,
            "fedprox_mu": FEDPROX_MU,
            "client_count_loaded": len(all_clients),
            "client_count_train": len(train_clients),
        },
        "optimization": {
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
            "grad_clip_norm": GRAD_CLIP_NORM,
        },
        "task": {
            "window_size": EXPECTED_WINDOW_SIZE,
            "horizon": int(metadata.get("horizon", SUPPORTED_HORIZON)),
            "target_column": metadata.get("target_column", SUPPORTED_TARGET_COLUMN),
            "feature_count": EXPECTED_FEATURE_COUNT,
        },
    }
    save_json(payload, output_path)


save_training_config(run_dir / "config.json", selected_device)

## 6. Model factory

Svaki klijent dobiva svjezu instancu istog `LSTMRegressor` modela i globalne tezine na pocetku lokalnog treninga.

In [26]:
def make_model() -> LSTMRegressor:
    return LSTMRegressor(
        input_size=EXPECTED_FEATURE_COUNT,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
    )


global_model = make_model().to(selected_device)
loss_fn = nn.MSELoss()
parameter_count = count_trainable_parameters(global_model)
model_state_bytes = estimate_model_state_bytes(global_model)
best_model_path = run_dir / "best_model.pt"

pd.DataFrame(
    [
        {"item": "parameter_count", "value": parameter_count},
        {"item": "model_state_bytes", "value": model_state_bytes},
        {"item": "hidden_size", "value": HIDDEN_SIZE},
        {"item": "num_layers", "value": NUM_LAYERS},
        {"item": "dropout", "value": DROPOUT},
    ]
)

,item,value
0,parameter_count,76417.0
1,model_state_bytes,305668.0
2,hidden_size,128.0
3,num_layers,1.0
4,dropout,0.0


## Evaluation helpers

Ako projekt u buducnosti izveze `evaluate_split`, notebook ga moze preuzeti. Trenutno se koristi ista notebook funkcija kao u centralized LSTM-u.

In [27]:
def predict_split(
    *,
    model: nn.Module,
    split_arrays: SplitArrays,
    batch_size: int,
    device: torch.device,
) -> np.ndarray:
    model.eval()
    loader = build_dataloader(split_arrays, batch_size, shuffle=False, device=device)
    y_predictions: list[np.ndarray] = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device=device, dtype=torch.float32, non_blocking=device.type == "cuda")
            y_batch = y_batch.to(device=device, dtype=torch.float32, non_blocking=device.type == "cuda")
            predictions = model(X_batch).squeeze(-1)
            if predictions.ndim == 0 and y_batch.numel() == 1:
                predictions = predictions.reshape_as(y_batch)
            if predictions.shape != y_batch.shape:
                raise RuntimeError(
                    f"Prediction shape {predictions.shape} does not match target shape {y_batch.shape}."
                )
            if not torch.isfinite(predictions).all():
                raise RuntimeError("Non-finite predictions encountered during evaluation.")
            y_predictions.append(predictions.cpu().numpy())

    return np.concatenate(y_predictions, axis=0)


if project_evaluate_split is not None:
    evaluate_split = project_evaluate_split
else:
    def evaluate_split(
        *,
        model: nn.Module,
        split_arrays: SplitArrays,
        batch_size: int,
        device: torch.device,
        loss_fn: nn.Module,
        method_name: str,
    ) -> dict[str, Any]:
        model.eval()
        loader = build_dataloader(split_arrays, batch_size, shuffle=False, device=device)
        total_loss = 0.0
        total_samples = 0
        y_predictions: list[np.ndarray] = []

        with torch.no_grad():
            for X_batch, y_batch in loader:
                X_batch = X_batch.to(device=device, dtype=torch.float32, non_blocking=device.type == "cuda")
                y_batch = y_batch.to(device=device, dtype=torch.float32, non_blocking=device.type == "cuda")

                predictions = model(X_batch).squeeze(-1)
                if predictions.ndim == 0 and y_batch.numel() == 1:
                    predictions = predictions.reshape_as(y_batch)
                if predictions.shape != y_batch.shape:
                    raise RuntimeError(
                        f"Prediction shape {predictions.shape} does not match target shape {y_batch.shape}."
                    )
                if not torch.isfinite(predictions).all():
                    raise RuntimeError("Non-finite predictions encountered during evaluation.")

                loss = loss_fn(predictions, y_batch)
                if not torch.isfinite(loss):
                    raise RuntimeError("Non-finite loss encountered during evaluation.")

                batch_size_actual = int(y_batch.shape[0])
                total_loss += float(loss.item()) * batch_size_actual
                total_samples += batch_size_actual
                y_predictions.append(predictions.cpu().numpy())

        if total_samples == 0:
            raise ValueError("Cannot evaluate an empty split.")

        y_pred = np.concatenate(y_predictions, axis=0)
        metrics = compute_regression_metrics(split_arrays.y, y_pred)
        prediction_frame = build_prediction_frame(
            t=split_arrays.t,
            station_ids=split_arrays.station_ids,
            y_true=split_arrays.y,
            y_pred=y_pred,
            method_name=method_name,
        )

        return {
            "loss": total_loss / total_samples,
            "metrics": metrics,
            "prediction_frame": prediction_frame,
        }


def empty_eval_result(method_name: str) -> dict[str, Any]:
    return {
        "loss": np.nan,
        "metrics": {"rmse": np.nan, "mae": np.nan, "r2": np.nan, "sample_count": 0},
        "prediction_frame": pd.DataFrame(
            columns=["t", "station_id", "y_true", "y_pred", "error", "abs_error", "method_name"]
        ),
    }


def evaluate_optional_split(
    *,
    model: nn.Module,
    split_arrays: SplitArrays | None,
    batch_size: int,
    device: torch.device,
    loss_fn: nn.Module,
    method_name: str,
) -> dict[str, Any]:
    if split_arrays is None or len(split_arrays.y) == 0:
        return empty_eval_result(method_name)
    return evaluate_split(
        model=model,
        split_arrays=split_arrays,
        batch_size=batch_size,
        device=device,
        loss_fn=loss_fn,
        method_name=method_name,
    )

## 7. Federated helper functions

FedAvg agregacija radi ponderirani prosjek po broju train uzoraka klijenta. Agregacija se radi na CPU-u radi jednostavnosti i ponovljivosti.

In [28]:
def clone_state_dict(state_dict: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    return {name: tensor.detach().cpu().clone() for name, tensor in state_dict.items()}


def state_dict_to_device(state_dict: dict[str, torch.Tensor], device: torch.device) -> dict[str, torch.Tensor]:
    return {name: tensor.detach().to(device) for name, tensor in state_dict.items()}


def fedavg_aggregate(
    client_states: list[dict[str, torch.Tensor]],
    client_sizes: list[int],
) -> dict[str, torch.Tensor]:
    if not client_states:
        raise ValueError("FedAvg requires at least one client state.")
    if len(client_states) != len(client_sizes):
        raise ValueError("client_states and client_sizes must have the same length.")

    sizes = np.asarray(client_sizes, dtype=np.float64)
    if (sizes <= 0).any():
        raise ValueError(f"All client sizes must be positive, got {client_sizes}.")
    total_size = float(sizes.sum())

    aggregated: dict[str, torch.Tensor] = {}
    for name in client_states[0].keys():
        first_tensor = client_states[0][name].detach().cpu()
        if torch.is_floating_point(first_tensor):
            accumulator_dtype = torch.float64 if first_tensor.dtype == torch.float64 else torch.float32
            accumulator = torch.zeros_like(first_tensor, dtype=accumulator_dtype, device="cpu")
            for state_dict, client_size in zip(client_states, sizes):
                local_tensor = state_dict[name].detach().cpu().to(dtype=accumulator_dtype)
                accumulator.add_(local_tensor, alpha=float(client_size / total_size))
            aggregated[name] = accumulator.to(dtype=first_tensor.dtype)
        else:
            aggregated[name] = first_tensor.clone()
    return aggregated

## 8. Local client training

Klijent u lokalnom treningu vidi samo svoj `train.npz`. Za FedProx se dodaje proksimalni clan prema zamrznutim globalnim parametrima.

In [29]:
def train_client(
    global_state_dict: dict[str, torch.Tensor],
    client: ClientDatasetBundle,
    model_fn,
    algorithm: str,
    local_epochs: int,
    batch_size: int,
    device: torch.device,
    lr: float,
    weight_decay: float,
    loss_fn: nn.Module,
    grad_clip_norm: float | None,
    fedprox_mu: float,
) -> tuple[dict[str, torch.Tensor], int, float]:
    num_samples = int(len(client.train.y))
    if num_samples == 0:
        raise ValueError(f"Client {client.client_id} has an empty train split.")

    normalized_algorithm = algorithm.strip().lower()
    if normalized_algorithm not in {"fedavg", "fedprox"}:
        raise ValueError("algorithm must be 'fedavg' or 'fedprox'.")

    local_model = model_fn().to(device)
    local_model.load_state_dict(state_dict_to_device(global_state_dict, device))
    local_model.train()

    global_parameters = {
        name: parameter.detach().clone().to(device)
        for name, parameter in local_model.named_parameters()
        if torch.is_floating_point(parameter)
    }
    for parameter in global_parameters.values():
        parameter.requires_grad_(False)

    optimizer = torch.optim.Adam(local_model.parameters(), lr=lr, weight_decay=weight_decay)
    loader = build_dataloader(client.train, batch_size, shuffle=True, device=device)

    total_loss = 0.0
    total_seen = 0
    for _ in range(local_epochs):
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device=device, dtype=torch.float32, non_blocking=device.type == "cuda")
            y_batch = y_batch.to(device=device, dtype=torch.float32, non_blocking=device.type == "cuda")

            optimizer.zero_grad(set_to_none=True)
            predictions = local_model(X_batch).squeeze(-1)
            if predictions.ndim == 0 and y_batch.numel() == 1:
                predictions = predictions.reshape_as(y_batch)
            if predictions.shape != y_batch.shape:
                raise RuntimeError(
                    f"Prediction shape {predictions.shape} does not match target shape {y_batch.shape} "
                    f"on {client.client_id}."
                )
            if not torch.isfinite(predictions).all():
                raise RuntimeError(f"Non-finite predictions encountered on {client.client_id}.")

            mse_loss = loss_fn(predictions, y_batch)
            if normalized_algorithm == "fedprox":
                proximal_term = torch.zeros((), device=device, dtype=mse_loss.dtype)
                for name, local_parameter in local_model.named_parameters():
                    if torch.is_floating_point(local_parameter):
                        proximal_term = proximal_term + torch.sum((local_parameter - global_parameters[name]) ** 2)
                loss = mse_loss + (fedprox_mu / 2.0) * proximal_term
            else:
                loss = mse_loss

            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss encountered on {client.client_id}.")

            loss.backward()
            if grad_clip_norm is not None and grad_clip_norm > 0.0:
                clip_grad_norm_(local_model.parameters(), grad_clip_norm)
            optimizer.step()

            batch_size_actual = int(y_batch.shape[0])
            total_loss += float(loss.item()) * batch_size_actual
            total_seen += batch_size_actual

    average_loss = total_loss / total_seen
    return clone_state_dict(local_model.state_dict()), num_samples, float(average_loss)

## 9. Global evaluation

Globalni train/val/test skupovi nastaju samo konkatenacijom istoimenih splitova. Prazni val/test splitovi pojedinih klijenata se ne ukljucuju.

In [30]:
method_name = f"{ALGORITHM}_lstm_{SCENARIO}"

global_split_overview = pd.DataFrame(
    [
        {"split": "train", "samples": int(len(global_train.y)), "X_shape": tuple(global_train.X.shape)},
        {"split": "val", "samples": int(len(global_val.y)), "X_shape": tuple(global_val.X.shape)},
        {
            "split": "test",
            "samples": 0 if global_test is None else int(len(global_test.y)),
            "X_shape": None if global_test is None else tuple(global_test.X.shape),
        },
    ]
)

global_split_overview

,split,samples,X_shape
0,train,110642,"(110642, 24, 19)"
1,val,6824,"(6824, 24, 19)"
2,test,30275,"(30275, 24, 19)"


## 10. Federated training loop

U svakoj rundi saljemo trenutni globalni `state_dict` svim trenirajucim klijentima, lokalno treniramo, agregiramo i validiramo globalni model.

In [ ]:
round_rows: list[dict[str, Any]] = []
best_val_rmse = float("inf")
best_round: int | None = None

global_state = clone_state_dict(global_model.state_dict())

for round_idx in range(1, NUM_ROUNDS + 1):
    client_states: list[dict[str, torch.Tensor]] = []
    client_sizes: list[int] = []
    client_losses: list[float] = []

    for client in train_clients:
        local_state, num_samples, local_loss = train_client(
            global_state_dict=global_state,
            client=client,
            model_fn=make_model,
            algorithm=ALGORITHM,
            local_epochs=LOCAL_EPOCHS,
            batch_size=BATCH_SIZE,
            device=selected_device,
            lr=LR,
            weight_decay=WEIGHT_DECAY,
            loss_fn=loss_fn,
            grad_clip_norm=GRAD_CLIP_NORM,
            fedprox_mu=FEDPROX_MU,
        )
        client_states.append(local_state)
        client_sizes.append(num_samples)
        client_losses.append(local_loss)

    global_state = fedavg_aggregate(client_states, client_sizes)
    global_model.load_state_dict(state_dict_to_device(global_state, selected_device))

    val_eval = evaluate_split(
        model=global_model,
        split_arrays=global_val,
        batch_size=BATCH_SIZE,
        device=selected_device,
        loss_fn=loss_fn,
        method_name=method_name,
    )
    val_metrics = val_eval["metrics"]
    train_loss_mean = float(np.mean(client_losses))
    train_loss_weighted = float(np.average(client_losses, weights=np.asarray(client_sizes, dtype=float)))
    communication_upload_bytes = model_state_bytes * len(train_clients)
    communication_download_bytes = model_state_bytes * len(train_clients)
    communication_total_bytes = communication_upload_bytes + communication_download_bytes
    communication_cumulative_bytes = communication_total_bytes * round_idx

    row = {
        "round": round_idx,
        "client_count": len(train_clients),
        "train_loss_mean": train_loss_mean,
        "train_loss_weighted": train_loss_weighted,
        "val_loss": float(val_eval["loss"]),
        "val_mae": float(val_metrics["mae"]),
        "val_rmse": float(val_metrics["rmse"]),
        "val_smape": float(val_metrics.get("smape", np.nan)),
        "val_mape": float(val_metrics.get("mape", np.nan)),
        "val_r2": float(val_metrics["r2"]),
        "communication_upload_mb": communication_upload_bytes / (1024 ** 2),
        "communication_download_mb": communication_download_bytes / (1024 ** 2),
        "communication_total_mb": communication_total_bytes / (1024 ** 2),
        "communication_cumulative_mb": communication_cumulative_bytes / (1024 ** 2),
    }
    round_rows.append(row)
    pd.DataFrame(round_rows).to_csv(run_dir / "round_metrics.csv", index=False)

    print(
        f"Round {round_idx}/{NUM_ROUNDS} | "
        f"train_loss={train_loss_weighted:.6f} | "
        f"val_rmse={row['val_rmse']:.6f} | "
        f"val_mae={row['val_mae']:.6f} | "
        f"val_r2={row['val_r2']:.6f}"
    )
    logger.info(
        "Round %d/%d | train_loss=%.6f | val_rmse=%.6f | val_mae=%.6f | val_r2=%.6f",
        round_idx,
        NUM_ROUNDS,
        train_loss_weighted,
        row["val_rmse"],
        row["val_mae"],
        row["val_r2"],
    )

    if row["val_rmse"] < best_val_rmse:
        best_val_rmse = row["val_rmse"]
        best_round = round_idx
        torch.save(global_state, best_model_path)
        logger.info("Saved new best model at round %d with val_rmse=%.6f", round_idx, best_val_rmse)

if best_round is None:
    raise RuntimeError("Training finished without producing a valid checkpoint.")

round_metrics = pd.DataFrame(round_rows)
round_metrics

2026-05-11 18:44:26,107 | INFO | Round 1/100 | train_loss=1576758.111503 | val_rmse=1010.747532 | val_mae=330.679216 | val_r2=-0.116787
2026-05-11 18:44:26,167 | INFO | Saved new best model at round 1 with val_rmse=1010.747532


Round 1/100 | train_loss=1576758.111503 | val_rmse=1010.747532 | val_mae=330.679216 | val_r2=-0.116787


2026-05-11 18:45:19,434 | INFO | Round 2/100 | train_loss=1570737.572976 | val_rmse=1009.161588 | val_mae=327.461372 | val_r2=-0.113286
2026-05-11 18:45:19,510 | INFO | Saved new best model at round 2 with val_rmse=1009.161588


Round 2/100 | train_loss=1570737.572976 | val_rmse=1009.161588 | val_mae=327.461372 | val_r2=-0.113286


2026-05-11 18:46:13,184 | INFO | Round 3/100 | train_loss=1566894.779856 | val_rmse=1007.802786 | val_mae=324.659396 | val_r2=-0.110290
2026-05-11 18:46:13,239 | INFO | Saved new best model at round 3 with val_rmse=1007.802786


...

2026-05-11 20:12:38,832 | INFO | Round 99/100 | train_loss=1261133.814675 | val_rmse=891.240313 | val_mae=250.477945 | val_r2=0.131690
2026-05-11 20:12:38,919 | INFO | Saved new best model at round 99 with val_rmse=891.240313


Round 99/100 | train_loss=1261133.814675 | val_rmse=891.240313 | val_mae=250.477945 | val_r2=0.131690


2026-05-11 20:13:32,748 | INFO | Round 100/100 | train_loss=1258287.610905 | val_rmse=890.094829 | val_mae=249.935214 | val_r2=0.133921
2026-05-11 20:13:32,822 | INFO | Saved new best model at round 100 with val_rmse=890.094829


Round 100/100 | train_loss=1258287.610905 | val_rmse=890.094829 | val_mae=249.935214 | val_r2=0.133921


,round,client_count,train_loss_mean,train_loss_weighted,val_loss,val_mae,val_rmse,val_smape,val_mape,val_r2,communication_upload_mb,communication_download_mb,communication_total_mb,communication_cumulative_mb
0,1,40,1.576759e+06,1.576758e+06,1.021611e+06,330.679216,1010.747532,NaN,NaN,-0.116787,11.660309,11.660309,23.320618,23.320618
1,2,40,1.570738e+06,1.570738e+06,1.018407e+06,327.461372,1009.161588,NaN,NaN,-0.113286,11.660309,11.660309,23.320618,46.641235
2,3,40,1.566895e+06,1.566895e+06,1.015666e+06,324.659396,1007.802786,NaN,NaN,-0.110290,11.660309,11.660309,23.320618,69.961853
3,4,40,1.563264e+06,1.563264e+06,1.012995e+06,322.699314,1006.476695,NaN,NaN,-0.107370,11.660309,11.660309,23.320618,93.282471
4,5,40,1.559659e+06,1.559659e+06,1.010319e+06,321.279371,1005.146205,NaN,NaN,-0.104444,11.660309,11.660309,23.320618,116.603088
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,40,1.269721e+06,1.269720e+06,8.004418e+05,252.086046,894.674125,NaN,NaN,0.124986,11.660309,11.660309,23.320618,2238.779297
96,97,40,1.266848e+06,1.266848e+06,7.983783e+05,251.547828,893.520169,NaN,NaN,0.127242,11.660309,11.660309,23.320618,2262.099915
97,98,40,1.263987e+06,1.263987e+06,7.963554e+05,251.002218,892.387481,NaN,NaN,0.129453,11.660309,11.660309,23.320618,2285.420532
98,99,40,1.261134e+06,1.261134e+06,7.943092e+05,250.477945,891.240313,NaN,NaN,0.131690,11.660309,11.660309,23.320618,2308.741150


## 11. Final evaluation

Nakon treninga ucitavamo najbolji checkpoint i racunamo finalne metrike na spojenim train, val i test splitovima.

In [ ]:
best_state = torch.load(best_model_path, map_location=selected_device)
global_model.load_state_dict(best_state)

train_eval = evaluate_optional_split(
    model=global_model,
    split_arrays=global_train,
    batch_size=BATCH_SIZE,
    device=selected_device,
    loss_fn=loss_fn,
    method_name=method_name,
)
val_eval = evaluate_optional_split(
    model=global_model,
    split_arrays=global_val,
    batch_size=BATCH_SIZE,
    device=selected_device,
    loss_fn=loss_fn,
    method_name=method_name,
)
test_eval = evaluate_optional_split(
    model=global_model,
    split_arrays=global_test,
    batch_size=BATCH_SIZE,
    device=selected_device,
    loss_fn=loss_fn,
    method_name=method_name,
)

metrics_summary = pd.DataFrame(
    [
        {"split": "train", **train_eval["metrics"]},
        {"split": "val", **val_eval["metrics"]},
        {"split": "test", **test_eval["metrics"]},
    ]
)
metrics_summary.to_csv(run_dir / "metrics_summary.csv", index=False)

estimated_total_upload_bytes = model_state_bytes * len(train_clients) * NUM_ROUNDS
estimated_total_download_bytes = model_state_bytes * len(train_clients) * NUM_ROUNDS
estimated_total_communication_bytes = estimated_total_upload_bytes + estimated_total_download_bytes

final_metrics = {
    "run_name": RUN_NAME,
    "algorithm": ALGORITHM,
    "scenario": SCENARIO,
    "best_round": best_round,
    "best_val_rmse": best_val_rmse,
    "metrics": {
        "train": train_eval["metrics"],
        "val": val_eval["metrics"],
        "test": test_eval["metrics"],
    },
    "loss": {
        "train": train_eval["loss"],
        "val": val_eval["loss"],
        "test": test_eval["loss"],
    },
    "sample_counts": {
        "train": int(len(global_train.y)),
        "val": int(len(global_val.y)),
        "test": 0 if global_test is None else int(len(global_test.y)),
    },
    "communication": {
        "model_state_bytes": model_state_bytes,
        "model_state_mb": model_state_bytes / (1024 ** 2),
        "total_rounds": NUM_ROUNDS,
        "train_client_count": len(train_clients),
        "estimated_total_upload_mb": estimated_total_upload_bytes / (1024 ** 2),
        "estimated_total_download_mb": estimated_total_download_bytes / (1024 ** 2),
        "estimated_total_communication_mb": estimated_total_communication_bytes / (1024 ** 2),
    },
}
save_json(final_metrics, run_dir / "final_metrics.json")

metrics_summary

/tmp/ipykernel_4602/2308897797.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  best_state = torch.load(best_model_path, map_location=selected_device)


,split,rmse,mae,r2,sample_count
0,train,1163.484236,369.597768,0.019445,110642
1,val,928.356080,268.174098,0.057862,6824
2,test,938.662612,274.734128,0.053886,30275


## 12. Save predictions

Test predikcije spremamo s kolonama trazenima za usporedbu metoda.

In [ ]:
def prediction_frame_for_submission(evaluation_result: dict[str, Any], method: str) -> pd.DataFrame:
    frame = evaluation_result["prediction_frame"].copy()
    if frame.empty:
        return pd.DataFrame(columns=["t", "station_id", "y_true", "y_pred", "method"])
    frame["method"] = method
    return frame.loc[:, ["t", "station_id", "y_true", "y_pred", "method"]]


predictions_test = prediction_frame_for_submission(test_eval, method_name)
predictions_test.to_csv(run_dir / "predictions_test.csv", index=False)

predictions_test.head()

,t,station_id,y_true,y_pred,method
0,2023-01-15T17:00:00.000000000,1001,0.0,4.295914,fedavg_lstm_iid
1,2023-01-15T18:00:00.000000000,1001,0.0,0.099844,fedavg_lstm_iid
2,2023-01-15T19:00:00.000000000,1001,0.0,0.752899,fedavg_lstm_iid
3,2023-01-15T20:00:00.000000000,1001,0.0,2.398837,fedavg_lstm_iid
4,2023-01-15T21:00:00.000000000,1001,0.0,1.448497,fedavg_lstm_iid


## 13. Plots

Osnovni grafovi za konvergenciju i test predikcije. Koristi se samo matplotlib.

In [ ]:
def save_line_plot(df: pd.DataFrame, x_col: str, y_col: str, title: str, y_label: str, output_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(df[x_col], df[y_col], marker="o", linewidth=1.8)
    ax.set_title(title)
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_label)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


save_line_plot(round_metrics, "round", "val_rmse", "Validation RMSE by round", "RMSE", plots_dir / "val_rmse_by_round.png")
save_line_plot(round_metrics, "round", "val_mae", "Validation MAE by round", "MAE", plots_dir / "val_mae_by_round.png")
save_line_plot(round_metrics, "round", "train_loss_weighted", "Weighted train loss by round", "MSE loss", plots_dir / "train_loss_by_round.png")

plot_frame = predictions_test.head(500).copy()
fig, ax = plt.subplots(figsize=(11, 4.8))
if not plot_frame.empty:
    ax.plot(plot_frame.index, plot_frame["y_true"], label="actual", linewidth=1.5)
    ax.plot(plot_frame.index, plot_frame["y_pred"], label="prediction", linewidth=1.5)
    ax.legend()
ax.set_title("Predictions vs actual - first 500 test samples")
ax.set_xlabel("sample")
ax.set_ylabel("target")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(plots_dir / "predictions_vs_actual_first_500.png", dpi=150)
plt.close(fig)

pd.DataFrame(
    [
        {"plot": path.name, "path": str(path)}
        for path in sorted(plots_dir.glob("*.png"))
    ]
)

,plot,path
0,predictions_vs_actual_first_500.png,/mnt/batch/tasks/shared/LS_root/mounts/cluster...
1,train_loss_by_round.png,/mnt/batch/tasks/shared/LS_root/mounts/cluster...
2,val_mae_by_round.png,/mnt/batch/tasks/shared/LS_root/mounts/cluster...
3,val_rmse_by_round.png,/mnt/batch/tasks/shared/LS_root/mounts/cluster...


## 14. Output structure

Svi artefakti ovog runa nalaze se u `data/processed/results/federated/{RUN_NAME}/`.

In [ ]:
artifact_paths = [
    run_dir / "config.json",
    run_dir / "client_summary.csv",
    run_dir / "round_metrics.csv",
    run_dir / "best_model.pt",
    run_dir / "metrics_summary.csv",
    run_dir / "final_metrics.json",
    run_dir / "predictions_test.csv",
    plots_dir,
]

pd.DataFrame(
    [
        {"artifact": path.name, "path": str(path), "exists": path.exists()}
        for path in artifact_paths
    ]
)

,artifact,path,exists
0,config.json,/mnt/batch/tasks/shared/LS_root/mounts/cluster...,True
1,client_summary.csv,/mnt/batch/tasks/shared/LS_root/mounts/cluster...,True
2,round_metrics.csv,/mnt/batch/tasks/shared/LS_root/mounts/cluster...,True
3,best_model.pt,/mnt/batch/tasks/shared/LS_root/mounts/cluster...,True
4,metrics_summary.csv,/mnt/batch/tasks/shared/LS_root/mounts/cluster...,True
5,final_metrics.json,/mnt/batch/tasks/shared/LS_root/mounts/cluster...,True
6,predictions_test.csv,/mnt/batch/tasks/shared/LS_root/mounts/cluster...,True
7,plots,/mnt/batch/tasks/shared/LS_root/mounts/cluster...,True


## 15. Final summary

Kratki izvjestaj koji se moze direktno prepisati u biljeske eksperimenta.

In [ ]:
total_train_samples = int(len(global_train.y))
total_val_samples = int(len(global_val.y))
total_test_samples = 0 if global_test is None else int(len(global_test.y))

print(f"Ucitano klijenata: {len(all_clients)}")
print(f"Ukupno uzoraka | train={total_train_samples} val={total_val_samples} test={total_test_samples}")
print(f"Rezultati su spremljeni u: {run_dir}")

metrics_summary

Ucitano klijenata: 40
Ukupno uzoraka | train=110642 val=6824 test=30275
Rezultati su spremljeni u: /mnt/batch/tasks/shared/LS_root/mounts/clusters/lgjuric2/code/Users/lgjuric/federated-ev-charging-forecasting/data/processed/results/federated/notebook_fedavg_iid_20260511T164513Z


,split,rmse,mae,r2,sample_count
0,train,1163.484236,369.597768,0.019445,110642
1,val,928.356080,268.174098,0.057862,6824
2,test,938.662612,274.734128,0.053886,30275
